<a href="https://colab.research.google.com/github/crystalloide/Big_Data/blob/master/handlab_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Traitement Big Data en Streaming avec Spark et Kafka
## Atelier Complet dans Google Colab

Ce notebook couvre l'intégralité du handlab :
- ✅ Configuration de l'environnement Spark
- ✅ Simulation d'un producteur Kafka
- ✅ Pipeline Spark Structured Streaming
- ✅ Filtrage des alertes
- ✅ Stockage en Parquet
- ✅ Inspection des résultats

**Durée totale : ~5 minutes**

## Étape 0 : Installation des Dépendances

In [ ]:
# Installer Java (requis pour Spark)
!apt-get update -qq
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null 2>&1

# Vérifier Java
!java -version

print("✅ Java installé")

In [ ]:
# Installer PySpark avec support Kafka
!pip install -q pyspark==3.5.0
!pip install -q kafka-python
!pip install -q pandas
!pip install -q matplotlib

print("✅ PySpark et dépendances installés")

In [ ]:
# Configuration des variables d'environnement
import os

os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-11-openjdk-amd64'

print("✅ Variables d'environnement configurées")

## Étape 1 : Initialisation de Spark

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, when,
    current_timestamp, count, collect_list, window
)
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd
from datetime import datetime, timedelta
import random
import json
import time
import shutil

# Créer la session Spark
spark = SparkSession.builder \
    .appName("StreamingAlertsDetection") \
    .master("local[*]") \
    .config("spark.sql.streaming.schemaInference", "true") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.driver.host", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("✅ SparkSession initialisée")
print(f"Spark Version: {spark.version}")

## Étape 2 : Simulation de Données Kafka

Puisque Colab n'a pas Kafka natif, nous allons simuler les données avec des fichiers JSON qui seront lus en streaming.

In [ ]:
# Créer les répertoires de travail
BASE_DIR = "/tmp/spark_streaming_labs"
DATA_SOURCE_DIR = f"{BASE_DIR}/data_source"  # Source de données simulées
ALERTS_DIR = f"{BASE_DIR}/alerte_logs"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoint_alerte_logs"

# Nettoyer et recréer les répertoires
for d in [DATA_SOURCE_DIR, ALERTS_DIR, CHECKPOINT_DIR]:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print(f"✅ Répertoires créés:")
print(f"   📂 {DATA_SOURCE_DIR}")
print(f"   📂 {ALERTS_DIR}")
print(f"   📂 {CHECKPOINT_DIR}")

In [ ]:
# Classe producteur de logs simulés

class LogProducer:
    """Produit des logs simulés au format JSON-like (stockés en fichiers pour Colab)."""

    def __init__(self, output_dir=DATA_SOURCE_DIR):
        self.output_dir = output_dir
        self.services = ["auth", "api", "database", "cache", "payment"]
        self.niveaux = ["INFO", "WARN", "ERROR", "CRITICAL"]
        self.messages_map = {
            "auth": [
                "Échec d'authentification",
                "Token invalide",
                "Timeout d'authentification",
                "Mot de passe incorrect"
            ],
            "api": [
                "Endpoint non trouvé",
                "Rate limit dépassé",
                "Erreur interne serveur",
                "Bad Request"
            ],
            "database": [
                "Connexion échouée",
                "Timeout requête",
                "Transaction échouée",
                "Deadlock détecté"
            ],
            "cache": [
                "Cache miss",
                "Erreur Redis",
                "Staleness détectée",
                "Eviction complète"
            ],
            "payment": [
                "Paiement refusé",
                "Timeout paiement",
                "Erreur gateway",
                "Solde insuffisant"
            ]
        }

    def generate_log(self):
        """Génère un log structuré."""
        service = random.choice(self.services)

        # Biaiser vers plus d'erreurs pour avoir des alertes intéressantes
        if random.random() < 0.3:  # 30% CRITICAL/ERROR
            niveau = random.choice(["CRITICAL", "ERROR"])
        else:
            niveau = random.choice(["INFO", "WARN"])

        message = random.choice(self.messages_map[service])

        # Timestamp récent
        delay_seconds = random.randint(-10, 60)
        timestamp = (datetime.utcnow() + timedelta(seconds=delay_seconds))

        return {
            "timestamp": timestamp.isoformat() + "Z",
            "service": service,
            "niveau": niveau,
            "message": message
        }

    def produce_batch(self, num_messages=100, batch_name="batch_001"):
        """
        Produit un batch de logs et l'enregistre en JSON.
        """
        logs = []

        for _ in range(num_messages):
            log_entry = self.generate_log()
            logs.append(log_entry)

        # Enregistrer le batch en JSON (ligne par ligne)
        output_file = f"{self.output_dir}/{batch_name}.json"
        with open(output_file, 'w') as f:
            for log in logs:
                f.write(json.dumps(log) + "\n")

        return len(logs), output_file

print("✅ Classe LogProducer définie")

In [ ]:
# Générer des données de test
producer = LogProducer()

print("📤 Génération de données de test...\n")

total_logs = 0
for i in range(1, 6):  # Générer 5 batches
    batch_name = f"batch_{i:03d}"
    num_logs, file_path = producer.produce_batch(num_messages=50, batch_name=batch_name)
    total_logs += num_logs
    print(f"✅ {batch_name}: {num_logs} logs → {file_path}")

print(f"\n📊 Total : {total_logs} logs générés")

In [ ]:
# Afficher un exemple de log
import json

with open(f"{DATA_SOURCE_DIR}/batch_001.json", 'r') as f:
    first_log = json.loads(f.readline())

print("📋 Exemple de log généré :")
print(json.dumps(first_log, indent=2, ensure_ascii=False))

## Étape 3 : Pipeline Spark Structured Streaming

Cette étape implémente les 5 étapes du handlab original :

In [ ]:
# Définir le schéma des logs

schema = StructType([
    StructField("timestamp", StringType(), True),
    StructField("service", StringType(), True),
    StructField("niveau", StringType(), True),
    StructField("message", StringType(), True)
])

print("✅ Schéma défini :")
for field in schema.fields:
    print(f"   - {field.name}: {field.dataType}")

In [ ]:
# ÉTAPE 1 : Lire le flux de données (simulation Kafka avec fichiers JSON)

print("\n📖 ÉTAPE 1 : Lecture de la source de données...")

df_raw = spark.readStream \
    .format("json") \
    .schema(schema) \
    .load(DATA_SOURCE_DIR)

print("✅ Source de données configurée (simulation fichiers JSON)")
print("   Note: En production, ce serait readStream.format('kafka')")

In [ ]:
# ÉTAPE 2 : Conversion et parsing des messages

print("\n🔄 ÉTAPE 2 : Parsing et conversion des messages...")

df_parsed = (df_raw
    .withColumn("timestamp",
        to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
    .filter(col("timestamp").isNotNull())
)

print("✅ Messages parsés et validés")
print(f"   - Timestamp converti en TimestampType")
print(f"   - Lignes avec timestamp null filtrées")

In [ ]:
# ÉTAPE 3 : Filtrage des événements d'alerte

print("\n🔍 ÉTAPE 3 : Filtrage des alertes critiques...")

df_alerte = df_parsed.filter(
    col("niveau").isin("ERROR", "CRITICAL")
)

print("✅ Filtre appliqué : niveau IN ('ERROR', 'CRITICAL')")

In [ ]:
# ÉTAPE 4 : Écriture en Parquet avec checkpointing

print("\n💾 ÉTAPE 4 : Configuration de l'écriture en Parquet...")

# Configuration de la requête streaming
query = (df_alerte.writeStream
    .format("parquet")
    .option("path", ALERTS_DIR)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .outputMode("append")
    .trigger(processingTime="5 seconds")
    .start()
)

print(f"✅ Configuration complète :")
print(f"   - Format: Parquet")
print(f"   - Sortie: {ALERTS_DIR}")
print(f"   - Checkpoint: {CHECKPOINT_DIR}")
print(f"   - Mode: append")
print(f"   - Trigger: 5 secondes")

In [ ]:
# ÉTAPE 5 : Lancement et monitoring du streaming

print("\n⏱️  ÉTAPE 5 : Lancement du streaming...\n")

start_time = time.time()
monitoring_duration = 20  # 20 secondes de monitoring

try:
    while query.isActive and (time.time() - start_time) < monitoring_duration:
        if query.lastProgress:
            progress = query.lastProgress
            num_input_rows = progress.get('numInputRows', 0)
            num_output_rows = progress.get('numOutputRows', 0)
            processing_time = progress.get('durationMs', {}).get('total', 0)

            elapsed = time.time() - start_time
            print(f"[{elapsed:.1f}s] Rows entrée: {num_input_rows} | Sorties (alertes): {num_output_rows} | Temps: {processing_time}ms")

        time.sleep(2)

except KeyboardInterrupt:
    pass

print(f"\n✅ Streaming terminé après {time.time() - start_time:.1f}s")

In [ ]:
# Arrêter le streaming proprement
query.stop()
print("⛔ Streaming arrêté")
print(f"📂 Checkpoint sauvegardé : {CHECKPOINT_DIR}")

## Étape 6 : Inspection et Analyse des Résultats

In [ ]:
# Lire les fichiers Parquet générés

print("\n" + "="*60)
print("📊 INSPECTION DES RÉSULTATS")
print("="*60)

if os.path.exists(ALERTS_DIR):
    # Lire tous les fichiers Parquet
    df_results = spark.read.parquet(ALERTS_DIR)

    total_alerts = df_results.count()
    print(f"\n📈 Total d'alertes trouvées : {total_alerts}")

    if total_alerts > 0:
        print(f"\n🏗️  Schéma des données :")
        df_results.printSchema()

        print(f"\n🔝 Dernières 10 alertes :")
        df_results.orderBy(df_results.timestamp.desc()).limit(10).show(truncate=False)
    else:
        print("⚠️  Aucune alerte trouvée")
else:
    print(f"❌ Répertoire {ALERTS_DIR} n'existe pas")

In [ ]:
# Statistiques détaillées par service

print("\n📊 Alertes par service :")
df_results.groupBy("service").agg(count("*").alias("count")).orderBy(col("count").desc()).show()

print("\n⚠️  Alertes par niveau de sévérité :")
df_results.groupBy("niveau").agg(count("*").alias("count")).show()

In [ ]:
# Exporter en DataFrame Pandas pour analyse

df_pandas = df_results.limit(100).toPandas()

print(f"\n📋 Vue d'ensemble des alertes (premiers 10 enregistrements) :")
print(df_pandas.head(10).to_string())

## Étape 7 : Visualisations et Analyse Avancée

In [ ]:
# Visualisation avec Matplotlib

import matplotlib.pyplot as plt
import numpy as np

# Alertes par service
alerts_by_service = df_results.groupBy("service").count().toPandas()
alerts_by_service = alerts_by_service.sort_values('count', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : Alertes par service
axes[0].bar(alerts_by_service['service'], alerts_by_service['count'], color='steelblue')
axes[0].set_title('Alertes par Service', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Nombre d\'alertes')
axes[0].set_xlabel('Service')
axes[0].tick_params(axis='x', rotation=45)

# Graphique 2 : Alertes par niveau
alerts_by_level = df_results.groupBy("niveau").count().toPandas()
colors = {'CRITICAL': 'red', 'ERROR': 'orange', 'WARN': 'yellow', 'INFO': 'green'}
color_list = [colors.get(level, 'gray') for level in alerts_by_level['niveau']]

axes[1].bar(alerts_by_level['niveau'], alerts_by_level['count'], color=color_list)
axes[1].set_title('Alertes par Niveau de Sévérité', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Nombre d\'alertes')
axes[1].set_xlabel('Niveau')

plt.tight_layout()
plt.show()

print("✅ Graphiques affichés")

In [ ]:
# Démonstration : Agrégations temporelles (bonus - Niveau avancé)

print("\n" + "="*60)
print("🎓 BONUS : Agrégations Temporelles (Niveau Avancé)")
print("="*60)

# Agrégation par fenêtre temporelle de 5 minutes
df_windowed = (df_results
    .groupBy(
        window(col("timestamp"), "5 minutes"),
        col("service")
    )
    .agg(count("*").alias("count"))
    .orderBy(col("window").desc())
)

print("\n📈 Alertes par fenêtre de 5 minutes et service :")
df_windowed.show(truncate=False)

## Résumé et Conclusions

In [ ]:
print("\n" + "="*70)
print("✅ RÉSUMÉ DE L'ATELIER")
print("="*70)

print("""
🎯 Objectifs Atteints :

1. ✅ Architecture Complète
   - Source de données (simulation Kafka)
   - Pipeline Spark Structured Streaming
   - Stockage en Parquet avec checkpointing

2. ✅ Transformations de Données
   - Parsing JSON
   - Conversion de types (timestamp)
   - Filtrage des alertes (ERROR, CRITICAL)

3. ✅ Concepts de Production
   - Checkpointing pour la résilience
   - Monitoring en temps réel
   - Output modes (append)
   - Trigger configuration

4. ✅ Analyse des Résultats
   - Statistiques par service
   - Statistiques par niveau de sévérité
   - Visualisations

📚 Concepts Maîtrisés :
   - Spark Structured Streaming
   - Schema validation
   - Stream processing patterns
   - Fault tolerance (checkpointing)

🚀 Évolutions Possibles (Niveau Avancé) :
   - Watermarking pour données tardives
   - Agrégations temporelles (time windows)
   - Webhooks pour alertes temps réel
   - Persistence en base de données
""")

print("="*70)
print(f"📂 Données générées dans : {ALERTS_DIR}")
print(f"🔄 Checkpoint préservé dans : {CHECKPOINT_DIR}")
print("="*70)

In [ ]:
# Afficher les fichiers générés

print("\n📦 Fichiers générés :")
print("\nDonnées source (fichiers JSON) :")
for f in sorted(os.listdir(DATA_SOURCE_DIR)):
    file_path = os.path.join(DATA_SOURCE_DIR, f)
    size = os.path.getsize(file_path)
    print(f"   - {f} ({size} bytes)")

print("\nAlertes (fichiers Parquet) :")
if os.path.exists(ALERTS_DIR):
    for f in sorted(os.listdir(ALERTS_DIR)):
        file_path = os.path.join(ALERTS_DIR, f)
        if os.path.isdir(file_path):
            print(f"   📁 {f}/")
        else:
            size = os.path.getsize(file_path)
            print(f"   - {f} ({size} bytes)")
else:
    print("   (Vide)")

## Ressources et Références

### Documentation Officielle
- **Spark Structured Streaming** : https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html
- **Kafka Documentation** : https://kafka.apache.org/documentation/
- **Databricks Academy** : https://academy.databricks.com/

### Concepts Clés Couverts
1. **Streaming** : Traitement continu de flux de données
2. **Micro-batching** : Traitement par petits batches (trigger-based)
3. **Stateful Processing** : Agrégations avec état (windowing)
4. **Checkpointing** : Sauvegarde de l'état pour la récupération
5. **Output Modes** : append, update, complete

### Prochaines Étapes
- Implémenter le watermarking pour gérer les données tardives
- Ajouter des webhooks pour les alertes temps réel
- Intégrer une base de données (PostgreSQL/MongoDB)
- Déployer en production avec orchestration (Airflow, Kubernetes)